In [ ]:
#!/usr/bin/env python
# coding: utf-8

# ## Maintain_Lakehouse
#
# Standalone maintenance notebook for compacting small files (OPTIMIZE) and
# removing stale files (VACUUM) across all Delta tables in the target lakehouse.
#
# OPTIMIZE: Compacts small Parquet files into larger ones. In Fabric, V-Order
#           is applied automatically during OPTIMIZE — no extra flag needed.
# VACUUM:   Removes files no longer referenced by the Delta log that are older
#           than the retention threshold. Default retention is 168 hours (7 days).
#
# Schedule: Run on a cadence that matches your ingestion frequency. Daily or
#           weekly is typical. OPTIMIZE is idempotent — running it on an already-
#           optimized table is a near-zero-cost no-op.

In [ ]:
# ── CELL 1: CONFIGURATION ──────────────────────────────────────────────────────

# This cell defines all configuration parameters for the maintenance process.
context = notebookutils.runtime.context

# --- Lakehouse Target ---
# Update WORKSPACE_NAME to match the Fabric workspace containing the lakehouse.
WORKSPACE_NAME = context.get("currentWorkspaceName")
LAKEHOUSE_NAME = context.get("defaultLakehouseName")

# --- Table Limit ---
# Set to -1 to process all discovered tables.
# Set to a positive integer to cap how many tables are processed (useful for testing).
TABLE_LIMIT = -1

# --- Vacuum Retention (hours) ---
# Files older than this threshold AND no longer referenced by the Delta log
# will be deleted by VACUUM. The Delta default minimum is 168 hours (7 days).
# Going below 168 requires disabling the safety check (not recommended in prod).
VACUUM_RETENTION_HOURS = 168

# --- Derived Path (do not edit) ---
SOURCE_ROOT = (
    f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_NAME}.Lakehouse/Tables"
)

print("--- Configuration ---")
print(f"  Workspace:        {WORKSPACE_NAME}")
print(f"  Lakehouse:        {LAKEHOUSE_NAME}")
print(f"  Table limit:      {'ALL' if TABLE_LIMIT == -1 else TABLE_LIMIT}")
print(f"  Vacuum retention: {VACUUM_RETENTION_HOURS} hours")
print(f"  Source root:      {SOURCE_ROOT}")
print("--- Cell 1 complete ---")

In [ ]:
# ── CELL 2: DISCOVER DELTA TABLES ──────────────────────────────────────────────

from notebookutils import mssparkutils
import py4j
import datetime

print("\n--- Cell 2: Discovering Delta tables (all schemas) ---")

all_tables = []

try:
    schemas = mssparkutils.fs.ls(SOURCE_ROOT)

    for schema in schemas:
        if not schema.isDir:
            continue

        schema_name = schema.name
        print(f"  Scanning schema: {schema_name}")

        try:
            tables = mssparkutils.fs.ls(schema.path)
        except Exception as e:
            print(f"  ⚠️  Could not scan schema '{schema_name}': {e}")
            continue

        for t in tables:
            if not t.isDir:
                continue

            # Confirm it's a Delta table by checking for _delta_log
            delta_log_path = f"{t.path}/_delta_log"
            try:
                mssparkutils.fs.ls(delta_log_path)
                spark_table_name = f"`{schema_name}`.`{t.name}`"
                all_tables.append(spark_table_name)
            except py4j.protocol.Py4JJavaError:
                # Expected for non-Delta folders — skip silently
                pass
            except Exception as e:
                print(f"  ⚠️  Error checking {schema_name}/{t.name}: {e}")

except Exception as e:
    print(f"❌ FATAL: Could not list source root '{SOURCE_ROOT}': {e}")
    raise

print(f"\nDiscovered {len(all_tables)} Delta tables across all schemas.")
print("--- Cell 2 complete ---")

In [ ]:
# ── CELL 3: OPTIMIZE + VACUUM ──────────────────────────────────────────────────

print("\n--- Cell 3: Running OPTIMIZE + VACUUM ---")

if TABLE_LIMIT == -1:
    tables_to_process = all_tables
else:
    tables_to_process = all_tables[:TABLE_LIMIT]

print(f"Processing {len(tables_to_process)} tables.\n")

# --- Result Tracking ---
results = []         # List of dicts with per-table outcomes
failed_tables = []   # Tables that failed either operation

start_time = datetime.datetime.now()

for table_name in tables_to_process:
    print(f"── {table_name} ──")
    result = {
        "table": table_name,
        "optimize_status": "SKIPPED",
        "vacuum_status": "SKIPPED",
        "files_removed": 0,
        "files_added": 0,
    }

    # --- OPTIMIZE ---
    try:
        print(f"  OPTIMIZE...", end=" ")
        optimize_result = spark.sql(f"OPTIMIZE {table_name}")

        # OPTIMIZE returns a DataFrame with a metrics struct.
        # Extract compaction stats from the first row.
        metrics_rows = optimize_result.collect()
        if metrics_rows:
            row = metrics_rows[0]
            try:
                metrics = row["metrics"]
                result["files_removed"] = metrics["numFilesRemoved"]
                result["files_added"] = metrics["numFilesAdded"]
                print(f"✅ (removed {metrics['numFilesRemoved']} files → {metrics['numFilesAdded']} files)")
            except Exception:
                # Metrics struct shape can vary by runtime — mark success anyway
                print(f"✅ (metrics not parseable — check Spark UI)")
        else:
            print(f"✅ (no rows returned — table may already be optimized)")

        result["optimize_status"] = "SUCCESS"

    except Exception as e:
        print(f"❌ FAILED: {e}")
        result["optimize_status"] = "FAILED"
        failed_tables.append(table_name)
        # Skip VACUUM if OPTIMIZE failed — table state is uncertain
        results.append(result)
        continue

    # --- VACUUM ---
    try:
        print(f"  VACUUM (retention={VACUUM_RETENTION_HOURS}h)...", end=" ")
        spark.sql(f"VACUUM {table_name} RETAIN {VACUUM_RETENTION_HOURS} HOURS")
        print(f"✅")
        result["vacuum_status"] = "SUCCESS"

    except Exception as e:
        print(f"❌ FAILED: {e}")
        result["vacuum_status"] = "FAILED"
        if table_name not in failed_tables:
            failed_tables.append(table_name)

    results.append(result)

end_time = datetime.datetime.now()
elapsed = end_time - start_time

print("\n--- Cell 3 complete ---")

In [ ]:
# ── CELL 4: SUMMARY ───────────────────────────────────────────────────────────

print("\n" + "=" * 80)
print("  MAINTENANCE SUMMARY")
print("=" * 80)

total_files_removed = sum(r["files_removed"] for r in results)
total_files_added = sum(r["files_added"] for r in results)
optimized_count = sum(1 for r in results if r["optimize_status"] == "SUCCESS")
vacuumed_count = sum(1 for r in results if r["vacuum_status"] == "SUCCESS")

print(f"\n  Tables processed:    {len(results)}")
print(f"  OPTIMIZE succeeded:  {optimized_count}")
print(f"  VACUUM succeeded:    {vacuumed_count}")
print(f"  Total files removed: {total_files_removed}")
print(f"  Total files created: {total_files_added}")
print(f"  Elapsed time:        {elapsed}")

# --- Per-Table Detail ---
print(f"\n  {'Table':<45} {'OPTIMIZE':<12} {'VACUUM':<12} {'Removed':>9} {'Added':>9}")
print(f"  {'-'*45} {'-'*12} {'-'*12} {'-'*9} {'-'*9}")

for r in results:
    print(
        f"  {r['table']:<45} "
        f"{r['optimize_status']:<12} "
        f"{r['vacuum_status']:<12} "
        f"{r['files_removed']:>9} "
        f"{r['files_added']:>9}"
    )

# --- Failed Tables ---
if failed_tables:
    print(f"\n  ⚠️  FAILED TABLES ({len(failed_tables)}):")
    for t in failed_tables:
        print(f"    - {t}")
else:
    print(f"\n  ✅ All tables completed successfully.")

print("\n" + "=" * 80)